In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

config = {
    "workspace_id"  : os.getenv("WORKSPACE_ID"),
    "lakehouse_id"  : os.getenv("LAKEHOUSE_ID"),
    "storage_url"   : os.getenv("STORAGE_URL")
}

BASE_PATH = f"abfss://{config['workspace_id']}@{config['storage_url']}/{config['lakehouse_id']}"

In [ ]:
file_name = '2016 Feb.csv'
processed_date = '2026-03-21'

In [ ]:
abfss = f"{BASE_PATH}/Files/raw"

In [ ]:
full_path = f'{abfss}/{file_name}'

In [ ]:
df = spark.read.csv(path=full_path, header=True, inferSchema=True)
display(df)

In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

from pyspark.sql.functions import lit, to_date, col, trim, date_format

if df.count() > 1:
    print("File contains data")
    print("Loading Data to landing zone...")
    
    # Fix dates here before saving to landing
    df_cleaned = df \
        .withColumn(
            "Order_Date",
            date_format(
                to_date(trim(col("Order Date")), "EEEE, MMMM dd, yyyy"), "yyyy-MM-dd"
            )
        ) \
        .withColumn(
            "Ship_Date",
            date_format(
                to_date(trim(col("Ship Date")), "EEEE, MMMM dd, yyyy"), "yyyy-MM-dd"
            )
        ) \
        .drop("Order Date", "Ship Date") \
        .withColumn("Processing_date", lit(processed_date))
    
    df_cleaned.write.mode("append") \
                .format('csv') \
                .option('header', True) \
                .partitionBy("Processing_date") \
            .save(f"{BASE_PATH}/Files/landing")
    # display(df_cleaned)
    print("Data loaded to landing zone successfully!")
else:
    print("File contained no data")